In [20]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 


In [21]:
CarsClean = pd.read_csv("Egypt_Used_Cars_clean.csv")
CarsClean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35394 entries, 0 to 35393
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   listing_id       35394 non-null  object 
 1   url              35394 non-null  object 
 2   make             35394 non-null  object 
 3   model            35394 non-null  object 
 4   year             35394 non-null  int64  
 5   kilometers       35394 non-null  float64
 6   price            35394 non-null  float64
 7   transmission     35394 non-null  object 
 8   fuel_type        35394 non-null  object 
 9   body_type        35394 non-null  object 
 10  exterior_color   35394 non-null  object 
 11  condition        35394 non-null  object 
 12  payment_options  35394 non-null  object 
 13  source           35394 non-null  object 
 14  age              35394 non-null  int64  
dtypes: float64(2), int64(2), object(11)
memory usage: 4.1+ MB


In [22]:

g = ["make", "model", "year"]

df = CarsClean[CarsClean.groupby(g).price.transform("size") >= 5].copy()
df["rel_price"] = df.price / df.groupby(g).price.transform("median")

c = df.groupby("exterior_color").agg(rel_price=("rel_price", "median"), n=("rel_price", "count")).reset_index()
c = c[c.n >= 30]   
px.bar(c.sort_values("rel_price", ascending=False), x="exterior_color", y="rel_price", text="n"
      ).add_hline(y=1, line_dash="dash").update_yaxes(range=[0.9, 1.1]).show()


In [23]:
order = CarsClean.groupby("body_type")["price"].median().sort_values(ascending=False).reset_index()
px.bar(order,x='body_type',y='price')

In [24]:
body_counts = (
    CarsDirty.groupby(["make", "model", "body_type"])
    .size()
    .reset_index(name="listings")
)

top_models = (
    body_counts.groupby(["make", "model"])["listings"]
    .sum()
    .nlargest(20)
    .index
)

plot_data = body_counts[
    body_counts.set_index(["make", "model"]).index.isin(top_models)
].copy()

plot_data["make_model"] = plot_data["make"] + " " + plot_data["model"]

fig = px.bar(
    plot_data,
    x="make_model",
    y="listings",
    color="body_type",
    title="Body Type Frequency by Make and Model (Top 20)",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [25]:
CarsClean["rel_price"] = CarsClean.price / CarsClean.groupby(["make", "model","year"]).price.transform("median")
px.box(
    df,
    y="rel_price",
    log_y=True,
    points="outliers",
    hover_data=["make", "model", "year", "price", "kilometers", "exterior_color"]
).show()

